In [ ]:
from pathlib import Path
import json
import pickle
import random
from collections import defaultdict

import numpy as np
import pandas as pd

In [ ]:
try:
    from tqdm.auto import tqdm
except ModuleNotFoundError:
    def tqdm(iterable=None, **kwargs):
        return iterable if iterable is not None else (lambda x: x)

In [ ]:
current_dir = Path.cwd().resolve()
PROJECT_ROOT = next(
    (path for path in [current_dir, *current_dir.parents] if (path / "data").exists() and (path / "notebooks").exists()),
    current_dir,
)

In [ ]:
DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
PROCESSED_DIR = DATA_DIR / "processed"
INDEX_DIR = PROCESSED_DIR / "indexes"
MODEL_DIR = PROCESSED_DIR / "models"

In [ ]:
for directory in [RAW_DIR, PROCESSED_DIR, INDEX_DIR, MODEL_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

In [ ]:
print(f"Project root: {PROJECT_ROOT}")
print(f"Raw dir: {RAW_DIR}")
print(f"Processed dir: {PROCESSED_DIR}")
print(f"Index dir: {INDEX_DIR}")
print(f"Model dir: {MODEL_DIR}")

In [ ]:
dataset_candidates = [
    PROCESSED_DIR / "news_dataset_df_prep2.pkl",
    PROCESSED_DIR / "news_dataset_df_prep3.pkl",
    DATA_DIR / "search_engine_corpus2.json",
    RAW_DIR / "news_dataset_df.pkl",
    RAW_DIR / "news_dataset.json",
]

In [ ]:
source_path = next((path for path in dataset_candidates if path.exists()), None)
if source_path is None:
    candidates_text = "\n".join(f"- {path}" for path in dataset_candidates)
    raise FileNotFoundError(
        "No local dataset was found. Run notebooks/Preprocessing.ipynb first, "
        "or place one of these files in the project:\n" + candidates_text
    )

if source_path.suffix == ".pkl":
    df_load = pd.read_pickle(source_path)
elif source_path.suffix == ".json":
    df_load = pd.read_json(source_path)
else:
    raise ValueError(f"Unsupported dataset format: {source_path}")

In [ ]:
if "id" not in df_load.columns:
    df_load = df_load.reset_index(drop=True)
    df_load["id"] = df_load.index.astype(int)

if "title" not in df_load.columns:
    df_load["title"] = ""

In [ ]:
if "combined_processed" not in df_load.columns:
    if {"title_processed", "content_processed"}.issubset(df_load.columns):
        df_load["combined_processed"] = (
            df_load["title_processed"].fillna("").astype(str)
            + " "
            + df_load["content_processed"].fillna("").astype(str)
        )
    elif "content" in df_load.columns:
        df_load["combined_processed"] = (
            df_load["title"].fillna("").astype(str)
            + " "
            + df_load["content"].fillna("").astype(str)
        ).str.lower()
        print("Warning: using raw title and content as combined_processed. Run Preprocessing.ipynb for better tokenized text.")
    else:
        raise ValueError(
            "Dataset must contain `combined_processed`, both `title_processed` and `content_processed`, "
            "or raw `content`."
        )

In [ ]:
df_load["combined_processed"] = df_load["combined_processed"].fillna("").astype(str)
df = df_load

print(f"Loaded DataFrame from: {source_path}")
print(f"Shape: {df_load.shape}")
print(f"Columns: {list(df_load.columns)}")

### Cuủ Duy

In [ ]:
def build_optimized_index(df_documents):
    inverted_index = defaultdict(list)
    doc_lengths = {}

    ids = df_documents["id"].values
    contents = df_documents["combined_processed"].values

    print("Building inverted index. This can take a few minutes.")
    for i in tqdm(range(len(df_documents))):
        doc_id = ids[i]
        tokens = str(contents[i]).split()
        doc_lengths[doc_id] = len(tokens)

        for token in set(tokens):
            inverted_index[token].append(doc_id)

    return inverted_index, doc_lengths


inverted_index, doc_lengths = build_optimized_index(df_load)

print(f"Built index with {len(inverted_index)} unique terms.")

### Verify Duy

In [ ]:
def convert_numpy_values(obj):
    if isinstance(obj, dict):
        return {str(k): convert_numpy_values(v) for k, v in obj.items()}
    if isinstance(obj, list):
        return [convert_numpy_values(item) for item in obj]
    if isinstance(obj, tuple):
        return [convert_numpy_values(item) for item in obj]
    if isinstance(obj, np.integer):
        return int(obj)
    if isinstance(obj, np.floating):
        return float(obj)
    return obj

In [ ]:
sample_keys = random.sample(list(inverted_index.keys()), min(10, len(inverted_index)))
inverted_index_samples = {key: inverted_index[key][:5] for key in sample_keys}

sample_index_path = INDEX_DIR / "inverted_index_samples.json"
full_index_path = INDEX_DIR / "inverted_index.json"
doc_lengths_path = INDEX_DIR / "doc_lengths.json"

In [ ]:
with open(sample_index_path, "w", encoding="utf-8") as f:
    json.dump(convert_numpy_values(inverted_index_samples), f, ensure_ascii=False, indent=2)

with open(full_index_path, "w", encoding="utf-8") as f:
    json.dump(convert_numpy_values(dict(inverted_index)), f, ensure_ascii=False, indent=2)

with open(doc_lengths_path, "w", encoding="utf-8") as f:
    json.dump(convert_numpy_values(doc_lengths), f, ensure_ascii=False, indent=2)

print(f"Saved sample index to: {sample_index_path}")
print(f"Saved full index to: {full_index_path}")
print(f"Saved document lengths to: {doc_lengths_path}")

In [ ]:
full_index_path = INDEX_DIR / "inverted_index.json"

with open(full_index_path, "r", encoding="utf-8") as f:
    inverted_index_data = json.load(f)

df_inverted_index = pd.DataFrame(inverted_index_data.items(), columns=["Term", "Doc IDs"])

print(f"Loaded inverted index from: {full_index_path}")
print(df_inverted_index.head(100))

### TF- IDF -Cosin

In [ ]:
try:
    from sklearn.feature_extraction.text import TfidfVectorizer
    from sklearn.metrics.pairwise import cosine_similarity
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError("Install scikit-learn first: pip install scikit-learn") from exc

In [ ]:
tfidf_corpus = df_load["combined_processed"].fillna("").astype(str).tolist()
vectorizer = TfidfVectorizer(
    token_pattern=r"(?u)\b\w+\b",
    lowercase=False,
    min_df=1,
    max_df=0.95,
)
tfidf_matrix = vectorizer.fit_transform(tfidf_corpus)

print(f"TF-IDF matrix shape: {tfidf_matrix.shape}")
print(f"Vocabulary size: {len(vectorizer.vocabulary_)}")

In [ ]:
def calculate_tf_for_query(tokens):
    tf = defaultdict(int)
    if not tokens:
        return tf

    total_terms = len(tokens)
    for term in tokens:
        tf[term] += 1
    for term in tf:
        tf[term] = tf[term] / total_terms

    return tf

In [ ]:
def search_documents(query_text, k, vectorizer, tfidf_matrix, df_documents):
    print(f"\nQuery: {query_text!r}")

    query_vector = vectorizer.transform([query_text])

    query_tokens = vectorizer.build_tokenizer()(query_text)
    query_tf = calculate_tf_for_query(query_tokens)

    query_term_details = []
    for term in query_tf:
        if term in vectorizer.vocabulary_:
            term_idx = vectorizer.vocabulary_[term]
            idf_val = vectorizer.idf_[term_idx]
            tf_val = query_tf[term]
            tfidf_val = tf_val * idf_val
            query_term_details.append({"Term": term, "TF": tf_val, "IDF": idf_val, "TF-IDF": tfidf_val})
        else:
            query_term_details.append({"Term": term, "TF": query_tf[term], "IDF": None, "TF-IDF": 0.0})

    query_details_df = pd.DataFrame(query_term_details)
    if not query_details_df.empty:
        print("\nQuery terms:")
        print(query_details_df.to_string(index=False))

    similarities = cosine_similarity(query_vector, tfidf_matrix).flatten()
    top_k_indices = similarities.argsort()[-k:][::-1]

    print(f"\nTop {k} similar documents:")
    results = []
    for rank, doc_idx in enumerate(top_k_indices, start=1):
        score = similarities[doc_idx]
        if score <= 0:
            continue

        row = df_documents.iloc[doc_idx]
        doc_id = row.get("id", doc_idx)
        doc_title = row.get("title", "")
        doc_content_processed = row.get("combined_processed", "")

        print(f"\nRank {rank} (Similarity: {score:.4f})")
        print(f"  Document ID: {doc_id}")
        print(f"  Title: {doc_title}")
        print(f"  Processed snippet: {str(doc_content_processed)[:150]}...")

        results.append({
            "rank": rank,
            "similarity_score": float(score),
            "doc_id": convert_numpy_values(doc_id),
            "title": str(doc_title),
            "processed_content_snippet": str(doc_content_processed)[:150] + "...",
        })

    if not results:
        print("No document has similarity score > 0.")

    return results

In [ ]:
query_example = "cướp tiệm vàng"
k_results = 5

search_results = search_documents(query_example, k_results, vectorizer, tfidf_matrix, df_load)

### Lưu mô hình TF-IDF và Vectorizer đã huấn luyện

In [ ]:
from scipy.sparse import save_npz

vectorizer_path = MODEL_DIR / "tfidf_vectorizer.pkl"
tfidf_matrix_path = MODEL_DIR / "tfidf_matrix.npz"

with open(vectorizer_path, "wb") as f:
    pickle.dump(vectorizer, f)

save_npz(tfidf_matrix_path, tfidf_matrix)

print(f"Saved TfidfVectorizer to: {vectorizer_path}")
print(f"Saved TF-IDF matrix to: {tfidf_matrix_path}")

In [ ]:
new_query = "bóng đá Việt Nam"
k_results = 5

search_documents(new_query, k_results, vectorizer, tfidf_matrix, df_load)